In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#****************************************************#
#       Downloading and Formatting the Data          #
#****************************************************#

In [ ]:
# Download Income Dataframe

income_df = pd.read_excel("/Users/ghadaelhusseini/Desktop/ZEW Interview Task/Income_Data_2023.xlsx")

In [ ]:
income_df

In [ ]:
income_df[income_df["Gebietseinheit"] == "Hamburg"]


In [ ]:
# Some fornatting and preprocessing for the Income dataframe

income_df = income_df.rename(columns={"Regional-schlüssel": "Kreis Code"})
income_df.loc[income_df['Gebietseinheit'] == 'Berlin', 'Kreis Code'] = '11000'   #---> I change the Berlin code to match the rest of the indicators


In [ ]:
# Download SGB_2 Dataframe
SGB_II_df = pd.read_excel('/Users/ghadaelhusseini/Desktop/ZEW Interview Task/SGB_II_2023 Data.xlsx')

In [ ]:
# Some formatting and preprocessing for the Income dataframe

SGB_II_df.loc[SGB_II_df['regionaleinheit'] == 'Hamburg', 'Kreis Code'] = '00002' #---> I change the Hamburg code to match the rest of the indicators

In [ ]:
# Download Educational_Attainment Dataframe

education_df = pd.read_excel('/Users/ghadaelhusseini/Desktop/ZEW Interview Task/School_Dropput__SHARE.xlsx')

In [ ]:
# Some formatting and preprocessing for the Education dataframe

education_df = education_df.rename(columns={"Unnamed: 0": "Kreis Code"})
education_df = education_df.rename(columns={"Insgesamt.1": "Insgesamt ohne ersten schulabschluss"})
education_df = education_df.rename(columns={"% share ohne Ersten Schulabschluss\t(Absolvierende/Abgehende allgemeinbildender Schulen nach dem Abschluss\t\t\t)": "Absolvierende/Abgehende allgemeinbildender Schulen nach dem Abschluss ohne Ersten Schulabschluss % share"})

education_df.loc[education_df['Kreis'] == ' Berlin', 'Kreis Code'] = '11000' #---> I change the Berlin code to match the rest of the indicators

In [ ]:
# Download Single_parents Dataframe

single_parents_df = pd.read_excel('/Users/ghadaelhusseini/Desktop/ZEW Interview Task/Alleinerziehenden Data.xlsx')

In [ ]:
# Reformat the Single parents df


df = single_parents_df.copy()

# Find the row that actually holds the single-parent data
mask = df.iloc[:, 0].astype(str).str.strip() == "Alleinerziehende Elternteile"
data_row = df[mask].iloc[0]

# Kreis blocks start at column 1, repeating every 4 columns:
# [Anzahl, flag, Prozent, flag]
records = []
col_names = df.columns
ncols = df.shape[1]

for start in range(1, ncols, 4):
    kreis_name = col_names[start]
    if not isinstance(kreis_name, str) or kreis_name.startswith("Unnamed"):
        continue
    anzahl_raw = data_row.iloc[start]
    prozent_raw = data_row.iloc[start + 2] if start + 2 < ncols else np.nan
    records.append({
        "Kreis_raw": kreis_name,
        "Anzahl": anzahl_raw,
        "Prozent": prozent_raw
    })

single_parents_clean = pd.DataFrame(records)

# Split "01001 Flensburg, Stadt" into AGS code + Kreis name
single_parents_clean["AGS"] = single_parents_clean["Kreis_raw"].str.extract(r"^(\d+)")
single_parents_clean["Kreis"] = single_parents_clean["Kreis_raw"].str.replace(r"^\d+\s*", "", regex=True)

# Convert German number format ('.' = thousands sep, ',' = decimal sep) to floats
def clean_number(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x in ("-", ".", "", "None", "nan"):
        return np.nan
    x = x.replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

single_parents_clean["Alleinerziehende_Anzahl"] = single_parents_clean["Anzahl"].apply(clean_number)
single_parents_clean["Alleinerziehende_Prozent"] = single_parents_clean["Prozent"].apply(clean_number)

single_parents_clean = single_parents_clean[
    ["AGS", "Kreis", "Alleinerziehende_Anzahl", "Alleinerziehende_Prozent"]
].dropna(subset=["Kreis"]).reset_index(drop=True)

single_parents_clean.head(10)

In [ ]:
# Some further formatting and preprocessing for the Education dataframe

single_parents_clean = single_parents_clean.rename(columns={"AGS": "Kreis Code"})
single_parents_clean.loc[single_parents_clean['Kreis'] == 'Hamburg, Freie und Hansestadt', 'Kreis Code'] = '00002' #---> I change the Hamburg code to match the rest of the indicators


In [ ]:
#***************************************************************#
#        Standardizing Kreis Names based on Kreis Code          #
#***************************************************************#

In [ ]:
import pandas as pd

# --- Step 1: standardize the Kreis code column in each dataframe ---
def clean_code(df, code_col="Kreis Code"):
    df = df.copy()
    df[code_col] = (
        df[code_col]
        .astype(str)
        .str.strip()
      #  .str.replace(r"\.0$", "", regex=True)   # in case it was read as float, e.g. "7320.0"
        .str.zfill(5)                            # pad to 5 digits with leading zeros
    )
    return df

sgb2_df = clean_code(SGB_II_df)
education_df = clean_code(education_df)
income_df = clean_code(income_df)
single_parents_df = clean_code(single_parents_clean)

# quick sanity check — should all show 5-character codes like '07320'
for name, df in [("SGB II", sgb2_df), ("Education", education_df),
                  ("Income", income_df), ("Single parents", single_parents_df)]:
    print(name, df["Kreis Code"].str.len().value_counts())

In [ ]:
#***************************************************************#
#        Merge All dataframes into one df                       #
#***************************************************************#

In [ ]:
final_df = sgb2_df[["Kreis Code", "regionaleinheit", "SGB II-Quote bis Altersgrenze (%) 2023"]].copy()

final_df = final_df.merge(
    education_df[["Kreis Code", "Absolvierende/Abgehende allgemeinbildender Schulen nach dem Abschluss ohne Ersten Schulabschluss % share"]],
    on="Kreis Code", how="left"
)

final_df = final_df.merge(
    income_df[["Kreis Code", "Primäreinkommen der privaten Haushalte einschl. der privaten Organisationen ohne Erwerbszweck 2023"]],
    on="Kreis Code", how="left"
)

final_df = final_df.merge(
    single_parents_df[["Kreis Code", "Alleinerziehende_Anzahl", "Alleinerziehende_Prozent"]],
    on="Kreis Code", how="left"
)

final_df = final_df.rename(columns={"Kreis code": "AGS"})

final_df.head(10)

In [ ]:
final_df

In [ ]:
# Quick Check to see if there are any missing values

missing_count = final_df["Absolvierende/Abgehende allgemeinbildender Schulen nach dem Abschluss ohne Ersten Schulabschluss % share"].isna().sum()
print(f"\nTotal rows with missing values: {missing_count}")

In [ ]:
#********************************************************************************#
#        Standardizing all the indicators using the Z-Score                      #
#********************************************************************************#

In [ ]:
import numpy as np

indicators = [
    "Primäreinkommen der privaten Haushalte einschl. der privaten Organisationen ohne Erwerbszweck 2023",
    "SGB II-Quote bis Altersgrenze (%) 2023",
    "Alleinerziehende_Prozent",
    "Absolvierende/Abgehende allgemeinbildender Schulen nach dem Abschluss ohne Ersten Schulabschluss % share"
]

income_col = indicators[0]

df_z = final_df.copy()

# Manual z-score standardization, NaN-safe:
# mean/std are computed ignoring NaNs, but any NaN input stays NaN in the output
for col in indicators:
    mean = final_df[col].mean(skipna=True)
    std = final_df[col].std(skipna=True)
    df_z[col] = (final_df[col] - mean) / std

# Reverse income so higher = more vulnerable, consistent with the other indicators
df_z[income_col] = -df_z[income_col]

# Index is NaN if ANY of the 4 indicators is missing for that region
df_z["socioeconomic_index"] = df_z[indicators].mean(axis=1, skipna=False)

# Sanity check: how many regions get excluded
n_missing = df_z["socioeconomic_index"].isna().sum()
print(f"{n_missing} regions excluded due to missing indicator(s)")

In [ ]:
# 1. Count how many NaN values are in the Socioeconomic column
missing_count = df_z["socioeconomic_index"].isna().sum()
print(f"Number of NaN values: {missing_count}")


In [ ]:
df_z

In [ ]:
df_z[df_z["Kreis Code"] == "06415"]

In [ ]:
# Checking what region names where missing --> originally Hamburg and Berlin were missing due to mismatch in code names, but that was fixed
missing_region_names = df_z[df_z["socioeconomic_index"].isna()]["regionaleinheit"].unique()
missing_region_codes = df_z[df_z["socioeconomic_index"].isna()]["Kreis Code"].unique()

print("Regions with missing socioeconomic index:")
print(missing_region_names)

print("\nRegion codes with missing socioeconomic index:")
print(missing_region_codes)


In [ ]:
#*************************************#
#       Plotting the Geo Map          #
#*************************************#

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import geopandas as gpd

gpkg = "/Users/ghadaelhusseini/Downloads/vg250_01-01.utm32s.gpkg.ebenen/vg250_ebenen_0101/DE_VG250.gpkg"

print(gpd.list_layers(gpkg))

In [ ]:
krs = gpd.read_file(
    gpkg,
    layer="vg250_krs"
)

print(krs.columns.tolist())

In [ ]:
merged_map = krs.merge(
    df_z,
    left_on="AGS",
    right_on="Kreis Code",
    how="left"
)

In [ ]:
print(
    "Kreise without indicator:",
    merged_map["socioeconomic_index"].isna().sum()
)

In [ ]:
missing = merged_map[
    merged_map["socioeconomic_index"].isna()
]

print(
    missing[["AGS", "GEN", "BEZ"]]
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 14))

# Plot the choropleth
merged_map.plot(
    column="socioeconomic_index",                 # <-- your indicator column
    cmap="YlOrRd",
    linewidth=0.4,
    edgecolor="white",
    legend=True,
    missing_kwds={
        "color": "black",
        "label": "No data"
    },
    ax=ax,
    legend_kwds={
        "label": "Socioeconomic vulnerability (higher = greater vulnerability)",
        "shrink": 0.7
    }
)

# --------------------------------------------------
# Add Kreis names
# --------------------------------------------------

for idx, row in merged_map.iterrows():

    point = row.geometry.representative_point()

    ax.annotate(
        text=row["GEN"],
        xy=(point.x, point.y),
        ha="center",
        va="center",
        fontsize=4,
        color="black"
    )

ax.set_axis_off()

plt.title(
    "Vulnerability Socioeconomic Indicator by Kreis, Germany",
    fontsize=16
)

plt.tight_layout()

# Save as high-quality vector PDF
plt.savefig(
    "germany_kreise_high_quality.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()